In [ ]:
derived nee

In [1]:
#!/usr/bin/env python
# coding: utf-8

import os
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# Suppress warnings
os.environ['PYTHONWARNINGS'] = 'ignore::FutureWarning'
warnings.filterwarnings("ignore", category=FutureWarning)

# ------------------ Paths & config ------------------
DATA_CSV        = "/explore/nobackup/people/spotter5/anna_v/v2/v2_model_training_final.csv"
BASE_OUT        = "/explore/nobackup/people/spotter5/anna_v/v2"
OUT_DIR         = os.path.join(BASE_OUT, "loocv_nolc", "nee_from_gpp_reco")
FIG_DIR         = os.path.join(OUT_DIR, "figures")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

# ------------------ Features (no land_cover) ------------------
FEATURE_COLS = [
    'EVI', 'NDVI', 'sur_refl_b01', 'sur_refl_b02', 'sur_refl_b03',
    'sur_refl_b07', 'NDWI', 'pdsi', 'srad', 'vap', 'vs', 'swe',
    'aet', 'pet', 'def',
    'bdod_0_100cm', 'cec_0_100cm', 'cfvo_0_100cm', 'clay_0_100cm',
    'nitrogen_0_100cm', 'ocd_0_100cm', 'phh2o_0_100cm', 'sand_0_100cm',
    'silt_0_100cm', 'soc_0_100cm', 'co2_cont', 'ALT',
    'lai', 'fpar', 'Percent_NonTree_Vegetation',
    'Percent_NonVegetated', 'Percent_Tree_Cover', 'sm_surface', 'sm_rootzone',
    'snow_cover', 'snow_depth', 'soil_temperature_level_1', 'soil_temperature_level_2',
    'soil_temperature_level_3', 'soil_temperature_level_4',
    'N_N_0_100cm', 'alpha_ALFA_0_100cm', 'crit_wilt_CRIT-WILT_0_100cm',
    'field_crit_FIELD-CRIT_0_100cm', 'ksat_Ksat_0_100cm', 'ormc_ORMC_0_100cm',
    'satfield_SAT-FIELD_0_100cm', 'stc_STC_0_100cm', 'wcavail_WCavail_0_100cm',
    'wcpf2_WCpF2_0_100cm', 'wcpf3_WCpF3_0_100cm', 'wcpf4_2_WCpF4-2_0_100cm',
    'wcres_WCres_0_100cm', 'wcsat_WCsat_0_100cm',
    'tmean_C',          # will create below
    'month'             # categorical
]
CATEGORICAL = ['month']

# ------------------ Model factory ------------------
def make_cb():
    return CatBoostRegressor(
        iterations=1200,
        learning_rate=0.01,
        depth=8,
        subsample=0.7,
        random_state=42,
        l2_leaf_reg=0.1,
        rsm=0.8,
        cat_features=CATEGORICAL,  # we pass a pandas DataFrame; names are OK
        verbose=0,
        allow_writing_files=False
    )

# ------------------ Metrics helper ------------------
def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    if mask.sum() == 0:
        return np.nan, np.nan, np.nan
    rmse = float(np.sqrt(mean_squared_error(y_true[mask], y_pred[mask])))
    mae  = float(mean_absolute_error(y_true[mask], y_pred[mask]))
    r2   = float(r2_score(y_true[mask], y_pred[mask]))
    return r2, rmse, mae

# ------------------ Main LOSO routine ------------------
def run_loso_gpp_reco_and_nee():
    print("--- Loading & preparing data ---")
    df = pd.read_csv(DATA_CSV)
    # EC only
    if 'flux_method' in df.columns:
        df = df[df['flux_method'] == 'EC'].copy()

    # Make sure essential columns exist
    required_cols = {'site_reference', 'year', 'month', 'gpp', 'reco', 'nee'}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    # Derived fields
    if {'tmmn', 'tmmx'}.issubset(df.columns):
        df['tmean_C'] = df[['tmmn', 'tmmx']].mean(axis=1)
    else:
        if 'tmean_C' not in df.columns:
            df['tmean_C'] = np.nan

    df['month'] = df['month'].astype(int)
    df['date'] = pd.to_datetime(df[['year', 'month']].assign(day=1))

    # Drop rows missing any required targets
    df = df.dropna(subset=['site_reference', 'gpp', 'reco', 'nee']).copy()

    # Build final feature matrix (only keep features present)
    present_feats = [c for c in FEATURE_COLS if c in df.columns]
    if 'month' not in present_feats:
        present_feats.append('month')
    X_all = df[present_feats].copy()

    # Ensure categorical dtypes
    for c in CATEGORICAL:
        if c in X_all.columns:
            X_all[c] = X_all[c].astype('category')

    sites = df['site_reference'].dropna().unique()
    sites = np.sort(sites)

    rows = []
    print(f"--- LOSO across {len(sites)} sites ---")

    for test_site in sites:
        print(f"  • Fold: left-out site = {test_site}")
        test_mask = df['site_reference'] == test_site
        train_mask = ~test_mask

        if test_mask.sum() == 0:
            continue

        X_train = X_all.loc[train_mask]
        X_test  = X_all.loc[test_mask]
        # Targets
        y_train_gpp  = df.loc[train_mask, 'gpp']
        y_train_reco = df.loc[train_mask, 'reco']

        # Train models
        model_gpp  = make_cb()
        model_reco = make_cb()
        model_gpp.fit(X_train, y_train_gpp)
        model_reco.fit(X_train, y_train_reco)

        # Predict on left-out site
        pred_gpp  = pd.Series(model_gpp.predict(X_test),  index=X_test.index, name='pred_gpp')
        pred_reco = pd.Series(model_reco.predict(X_test), index=X_test.index, name='pred_reco')

        # Derived NEE = modeled RECO − modeled GPP
        derived_nee = pred_reco - pred_gpp
        derived_nee.name = 'derived_nee'

        # Collect outputs for this fold
        fold = pd.DataFrame({
            'site_reference': df.loc[test_mask, 'site_reference'],
            'date': df.loc[test_mask, 'date'],
            'year': df.loc[test_mask, 'year'],
            'month': df.loc[test_mask, 'month'],
            'observed_nee': df.loc[test_mask, 'nee'],
            'pred_gpp': pred_gpp,
            'pred_reco': pred_reco,
            'derived_nee': derived_nee
        }, index=X_test.index).sort_values('date')

        rows.append(fold)

    if not rows:
        print("No folds produced predictions. Exiting.")
        return

    # Pooled predictions across all left-out sites
    pooled = pd.concat(rows, axis=0).reset_index(drop=True)

    # Clean infinities
    pooled = pooled.replace([np.inf, -np.inf], np.nan).dropna(subset=['observed_nee', 'derived_nee'])

    # Save pooled predictions
    pooled_csv = os.path.join(OUT_DIR, "nee_loso_pooled_predictions.csv")
    pooled.to_csv(pooled_csv, index=False)
    print(f"Saved pooled predictions to: {pooled_csv}")

    # Compute pooled metrics: derived NEE vs observed nee
    r2, rmse, mae = compute_metrics(pooled['observed_nee'], pooled['derived_nee'])
    print(f"--- Pooled metrics (Derived NEE vs Observed NEE) ---")
    print(f"R² = {r2:.4f} | RMSE = {rmse:.4f} | MAE = {mae:.4f}")

    # -------- Density (hexbin) plot instead of scatter --------
    fig, ax = plt.subplots(figsize=(7, 7))
    x = pooled['observed_nee'].values
    y = pooled['derived_nee'].values

    # Limits + padding
    lo = np.nanmin([np.nanmin(x), np.nanmin(y)])
    hi = np.nanmax([np.nanmax(x), np.nanmax(y)])
    span = hi - lo if np.isfinite(hi - lo) and (hi - lo) > 0 else 1.0
    pad = 0.05 * span
    x_min, x_max = lo - pad, hi + pad
    y_min, y_max = lo - pad, hi + pad
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)

    # Hexbin density (log-scaled counts)
    hb = ax.hexbin(
        x, y,
        gridsize=80,
        bins='log',
        mincnt=1
    )
    cbar = fig.colorbar(hb, ax=ax)
    cbar.set_label('log10(N points)')

    # 1:1 line
    ax.plot([x_min, x_max], [y_min, y_max], 'k-', linewidth=1.5)

    ax.set_title("Derived NEE (RECO−GPP) vs Observed NEE (LOSO)")
    ax.set_xlabel("Observed NEE")
    ax.set_ylabel("Derived NEE (modeled RECO − modeled GPP)")
    ax.grid(True, linestyle='--', alpha=0.3)

    # Annotation (lower right)
    annot = f"R² = {r2:.2f}\nRMSE = {rmse:.2f}\nMAE = {mae:.2f}"
    ax.text(
        0.97, 0.03, annot, transform=ax.transAxes,
        fontsize=11, va='bottom', ha='right',
        bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.75)
    )

    fig_path = os.path.join(FIG_DIR, "nee_loso_obs_vs_derived_density.png")
    plt.savefig(fig_path, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"Saved density plot to: {fig_path}")

if __name__ == "__main__":
    run_loso_gpp_reco_and_nee()


--- Loading & preparing data ---


/explore/nobackup/people/spotter5/temp_dir/ipykernel_2979199/2119692777.py:76: DtypeWarning: Columns (135) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(DATA_CSV)


--- LOSO across 153 sites ---
  • Fold: left-out site = Abisko Stordalen birch forest_tower
  • Fold: left-out site = Adventdalen_SJ-Adv_tower
  • Fold: left-out site = Alberta - Western Peatland - LaBiche River,Black Spruce,Larch Fen_CA-WP1_tower
  • Fold: left-out site = Alberta - Western Peatland - Poor Fen (Sphagnum moss)_CA-WP2_tower
  • Fold: left-out site = Alberta - Western Peatland - Rich Fen  (Carex)_CA-WP3_tower
  • Fold: left-out site = Anaktuvuk River Moderate Burn_US-An2_tower
  • Fold: left-out site = Anaktuvuk River Severe Burn_US-An1_tower
  • Fold: left-out site = Anaktuvuk River Unburned_US-An3_tower
  • Fold: left-out site = Andoya_NO-And_tower
  • Fold: left-out site = Atqasuk_US-Atq_tower
  • Fold: left-out site = Attawapiskat River Bog_CA-ARB_tower
  • Fold: left-out site = Attawapiskat River Fen_CA-ARF_tower
  • Fold: left-out site = Barrow-BEO_US-Beo_tower
  • Fold: left-out site = Barrow-BES_US-Bes_tower
  • Fold: left-out site = Bayelva, Spitsbergen_SJ-Blv_to